# 건강 상담 글: 문서 길이와 청킹 실험 벡터 (Colab GPU)

앱의 미세조정 ko-sroberta(`blanden77/ko-sroberta-dograg-b`)는 문서를 128토큰에서 자른다. 상담 글의 52%가 잘린다. `scripts/compare_doc_lengths.py`가 비교할 문서 벡터를 만든다.

- `len128`: 지금과 같은 128토큰 (검산용)
- `len256`, `len512`: 길이 제한만 늘림
- `chunk128`: 126토큰 창을 63토큰씩 밀며 나눈 조각 (특수 토큰 포함 128). `chunk128_owner.npy`는 조각마다 원래 글의 행 번호

- 런타임: GPU(T4 이상)
- HF 토큰: Colab Secrets의 `HF_TOKEN` (없으면 실행 중 입력)
- 결과: 비공개 데이터셋 `blanden77/dograg-data`의 `experiments` 브랜치 `doclen/` 아래에 올린다(벡터만, 문서 본문은 올리지 않음). 행 순서 = `df.csv` 행 번호 = 앱의 문서 ID.

In [ ]:
!pip -q install -U sentence-transformers huggingface_hub
import torch; print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU 없음: 런타임 유형을 GPU로 바꾸세요')

In [ ]:
import getpass
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
HF_TOKEN = (HF_TOKEN or getpass.getpass('HF_TOKEN (영문 입력 상태에서 붙여 넣기): ')).strip()
# A token typed with the Korean input mode on turns into Hangul and fails later as UnicodeEncodeError.
if not HF_TOKEN.isascii() or not HF_TOKEN.startswith('hf_'):
    raise ValueError(f'HF_TOKEN이 hf_로 시작하는 영문·숫자 값이 아닙니다(길이 {len(HF_TOKEN)}, 한글 포함: {not HF_TOKEN.isascii()}). '
                     'Hugging Face 토큰을 복사해 Colab Secrets의 HF_TOKEN에 붙여 넣으세요.')
print('HF_TOKEN ok, length', len(HF_TOKEN))

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download
REPO, REVISION = 'blanden77/dograg-data', 'data-v3'  # src/private_data.py
path = hf_hub_download(REPO, 'data/df.csv', repo_type='dataset', revision=REVISION, token=HF_TOKEN)
df = pd.read_csv(path)
# The app's documents are qa.input with carriage returns removed (19,206 of 19,206 match the Chroma texts).
CR = chr(13)
texts = df['qa.input'].astype(str).str.replace(CR, '', regex=False).tolist()
print(len(texts))

In [ ]:
import time, numpy as np
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('blanden77/ko-sroberta-dograg-b', device='cuda', token=HF_TOKEN)

def encode(items, max_len):
    model.max_seq_length = max_len
    started = time.time()
    vectors = model.encode(items, batch_size=128, normalize_embeddings=True, show_progress_bar=True,
                           convert_to_numpy=True).astype(np.float32)
    print(max_len, vectors.shape, f'{time.time() - started:.0f}s')
    return vectors

for max_len in (128, 256, 512):
    np.save(f'len{max_len}.npy', encode(texts, max_len))

In [ ]:
# Sliding windows over the model's own tokens; each window is cut from the original text by character offsets.
WINDOW, STRIDE = 126, 63  # 126 + [CLS] + [SEP] = the model's 128
chunks, owner = [], []
for row, text in enumerate(texts):
    offsets = model.tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)['offset_mapping']
    if len(offsets) <= WINDOW:
        chunks.append(text); owner.append(row)
        continue
    start = 0
    while True:
        end = min(start + WINDOW, len(offsets))
        chunks.append(text[offsets[start][0]:offsets[end - 1][1]]); owner.append(row)
        if end == len(offsets):
            break
        start += STRIDE
print(len(texts), 'documents ->', len(chunks), 'chunks')
np.save('chunk128.npy', encode(chunks, 128))
np.save('chunk128_owner.npy', np.array(owner, dtype=np.int32))

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
api.create_branch(REPO, repo_type='dataset', branch='experiments', exist_ok=True)
for name in ('len128', 'len256', 'len512', 'chunk128', 'chunk128_owner'):
    api.upload_file(path_or_fileobj=f'{name}.npy', path_in_repo=f'doclen/{name}.npy',
                    repo_id=REPO, repo_type='dataset', revision='experiments')
    print('uploaded', name)